# 🔍 Project: Biomedical RAG Chatbot

This notebook walks through building a Retrieval-Augmented Generation (RAG) chatbot for answering medical questions using BioMistral-7B.

In [34]:
!pip install sentence-transformers faiss-cpu bitsandbytes accelerate

## 📦 Step 1: Load & Preprocess Dataset
We use MedQuad (Q&A pairs) for fine-tuning and BioASQ PubMed snippets as the retriever corpus.

In [39]:
import json

with open("training13b.json", "r") as f:
    bioasq_raw = json.load(f)
bioasq_chunks = []

for q in bioasq_raw["questions"]:
    for snippet in q.get("snippets", []):
        text = snippet.get("text", "").strip()
        if len(text) > 50:
            chunk = {
                "id": f"{snippet.get('document', 'unknown')}_{len(bioasq_chunks)}",
                "text": text
            }
            bioasq_chunks.append(chunk)

print(f"✅ Extracted {len(bioasq_chunks)} usable chunks.")

documents = [chunk["text"] for chunk in bioasq_chunks]
doc_ids = [chunk["id"] for chunk in bioasq_chunks]
embeddings = embedder.encode(documents, convert_to_numpy=True, show_progess_bar=True)


✅ Extracted 67293 usable chunks.


In [40]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Generate embeddings for all chunks
embeddings = embedder.encode(documents, convert_to_numpy=True, show_progress_bar=True)

Batches:   0%|          | 0/2103 [00:00<?, ?it/s]

In [41]:
import faiss
import numpy as np

dimension = embeddings.shape[1]  # 384 for MiniLM
index = faiss.IndexFlatL2(dimension)  # L2 = Euclidean distance
index.add(embeddings)

In [42]:
def retrieve_top_k(query, k=5):
    query_embedding = embedder.encode([query])
    D, I = index.search(np.array(query_embedding), k)
    return [(doc_ids[i], documents[i]) for i in I[0]]

## 🧠 Step 2: Load BioMistral Model
We load BioMistral with 4-bit quantization for efficient generation.

In [43]:
query = "What are the symptoms of asthma?"
results = retrieve_top_k(query)

for doc_id, text in results:
    print(f"\n📄 {doc_id}:\n{text}")


📄 http://www.ncbi.nlm.nih.gov/pubmed/37260069_60908:
To determine the association between serum periostin levels and asthma control in children.

📄 http://www.ncbi.nlm.nih.gov/pubmed/25671117_9135:
Patients with severe asthma or COPD have often a suboptimal symptom control due to inadequate treatment.

📄 http://www.ncbi.nlm.nih.gov/pubmed/34606305_57662:
Tezepelumab in adults and adolescents with severe, uncontrolled asthma.

📄 http://www.ncbi.nlm.nih.gov/pubmed/25037608_60955:
BACKGROUND: Recent studies recommend periostin as a systemic biomarker of eosinophilic airway inflammation to predict responses to novel treatments that targets eosinophilic TH2-driven inflammation in asthmatic patients.OBJECTIVE: To investigate the clinical implications of serum periostin levels in patients with aspirin-exacerbated respiratory disease (AERD) based on its overlapping TH2-mediated pathogenesis with the eosinophilic asthma.METHODS: Serum periostin levels were measured by human periostin enzyme-li

In [44]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Load BioMistral
model_name = "BioMistral/BioMistral-7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True)
model.eval()

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): Mist

In [45]:
def build_prompt(query, context_chunks):
    context = "\n\n".join([chunk for _, chunk in context_chunks])
    return f"""You are a helpful medical assistant.

Context:
{context}

Question: {query}
Answer:"""

In [46]:
def generate_answer(query):
    top_chunks = retrieve_top_k(query)
    prompt = build_prompt(query, top_chunks)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [47]:
print(generate_answer("What are the symptoms of asthma?"))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are a helpful medical assistant.

Context:
To determine the association between serum periostin levels and asthma control in children.

Patients with severe asthma or COPD have often a suboptimal symptom control due to inadequate treatment.

Tezepelumab in adults and adolescents with severe, uncontrolled asthma.

BACKGROUND: Recent studies recommend periostin as a systemic biomarker of eosinophilic airway inflammation to predict responses to novel treatments that targets eosinophilic TH2-driven inflammation in asthmatic patients.OBJECTIVE: To investigate the clinical implications of serum periostin levels in patients with aspirin-exacerbated respiratory disease (AERD) based on its overlapping TH2-mediated pathogenesis with the eosinophilic asthma.METHODS: Serum periostin levels were measured by human periostin enzyme-linked immunosorbent assay (ELISA) in serum samples from 277 adults with asthma. Serum periostin levels were compared between patients with AERD and aspirin tolerant a

## 🔎 Step 3: Embed & Index Corpus for Retrieval
Using Sentence-BERT and FAISS, we embed and index biomedical chunks for retrieval.

##Filtering Duplicates in Retrieved Chunks


In [48]:
unique_chunks = list(dict.fromkeys([text for _, text in retrieve_top_k(query)]))
context = "\n\n".join(unique_chunks)


In [49]:
!pip install gradio
import gradio as gr
gr.Interface(fn=generate_answer, inputs="text", outputs="text", title="🩺 Biomedical RAG Chatbot").launch()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e12cdd56da0dc65516.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [50]:
def rag_chatbot(query):
    # Step 1: Retrieve top-k relevant chunks
    top_chunks = retrieve_top_k(query, k=5)

    # Remove duplicates
    unique_context = "\n\n".join(dict.fromkeys([text for _, text in top_chunks]))

    # Step 2: Build prompt
    prompt = f"""You are a helpful medical assistant.

Context:
{unique_context}

Question: {query}
Answer:"""

    # Step 3: Generate answer
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer


## 🔗 Step 4: RAG Pipeline – Retrieval + Generation
Top-k relevant chunks are retrieved, passed to BioMistral for grounded QnA.

In [51]:
import gradio as gr

gr.Interface(
    fn=rag_chatbot,
    inputs=gr.Textbox(lines=2, placeholder="Ask a medical question..."),
    outputs="text",
    title="🩺 Biomedical QnA Chatbot (RAG + BioMistral)",
    description="Powered by FAISS retrieval + BioMistral-7B"
).launch()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cf59e4718529431e78.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 🧪 Step 5: Testing the Chatbot
Ask medical queries and observe how the system answers with context.

## ✅ Summary
You now have a full RAG chatbot that combines domain-specific retrieval and LLM-powered reasoning.